In [12]:
import numpy as np


In [13]:
np.random.seed(42)

# Example: 3 features (area, bedrooms, age)
X = np.random.rand(300, 3)

# True weights (unknown to model)
true_w = np.array([200, 50000, -300])
y = X @ true_w + np.random.randn(300) * 1000


In [14]:
def split_data(X, y, num_clients=3):
    client_data = []
    size = len(X) // num_clients

    for i in range(num_clients):
        start = i * size
        end = (i + 1) * size
        client_data.append((X[start:end], y[start:end]))

    return client_data

clients = split_data(X, y, num_clients=3)


In [15]:
class LinearRegression:
    def __init__(self, n_features):
        self.weights = np.zeros(n_features)

    def predict(self, X):
        return X @ self.weights

    def train(self, X, y, lr=0.01, epochs=50):
        n = len(X)

        for _ in range(epochs):
            preds = self.predict(X)
            gradient = (1/n) * (X.T @ (preds - y))
            self.weights -= lr * gradient


In [16]:
def federated_average(client_weights):
    return np.mean(client_weights, axis=0)

In [17]:
num_features = X.shape[1]
global_weights = np.zeros(num_features)

rounds = 10

for r in range(rounds):
    client_weights = []

    for client_X, client_y in clients:
        # Client receives global model
        local_model = LinearRegression(num_features)
        local_model.weights = global_weights.copy()

        # Local training
        local_model.train(client_X, client_y)

        # Send weights to server
        client_weights.append(local_model.weights)

    # Server aggregates
    global_weights = federated_average(client_weights)

    print(f"Round {r+1} Global Weights:", global_weights)

Round 1 Global Weights: [5124.47349656 7129.65591911 4811.4070385 ]
Round 2 Global Weights: [ 8317.5923821 12257.1572007  7801.0225738]
Round 3 Global Weights: [10238.20462155 16040.32139255  9590.58082531]
Round 4 Global Weights: [11322.94736023 18916.1256437  10592.11392176]
Round 5 Global Weights: [11861.08884968 21175.04504074 11078.58109109]
Round 6 Global Weights: [12044.16742793 23010.35931065 11230.71353123]
Round 7 Global Weights: [11998.91444595 24550.8570228  11168.08414251]
Round 8 Global Weights: [11809.09014587 25882.52798941 10969.71428136]
Round 9 Global Weights: [11529.96597824 27062.9519475  10687.7408715 ]
Round 10 Global Weights: [11197.92962598 28130.84294314 10356.48075613]


In [18]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

final_preds = X @ global_weights
print("Final MSE:", mse(y, final_preds))

Final MSE: 60548053.774800286
